# Enrollment Agent Development Workbook
## Build and test one Q&M Enrollment Agent independently, then map it to the application

### Step 0 — Purpose and scope
⬜ **NOTEBOOK ONLY**

This adapted workbook runs in the existing local Q&M project environment, not in Google Colab. It recreates and proves Mr. Wee's independent-development sequence against the existing production Enrollment Agent: **Tool → direct tool test → Agent → standalone Task → one-agent Crew → `kickoff_async()` → FakeLLM test → SkillSpec → framework mapping**. It is evidence of the method and current implementation, not a claim that the production Agent was historically created in this notebook. The working Flask/React application continues to run from the Python modules under `backend/`; this notebook is not an application runtime dependency.

---
### Step 1 — Safe local setup
⬜ **NOTEBOOK ONLY**

Run this adapted workbook from the repository root with the existing local project virtual environment; it is intentionally not a Google Colab installation workflow. These cells do not install packages, load `.env`, request a key, initialise Flask, open SQLAlchemy, create invoices or send email. The scripted LLM and network guard make the exercise offline.

In [ ]:
import os
import sys
from pathlib import Path

os.environ.setdefault("CREWAI_DISABLE_TELEMETRY", "true")
os.environ.setdefault("OTEL_SDK_DISABLED", "true")

candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(path for path in candidates if (path / "backend" / "agents").is_dir())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print("Project imports ready; no environment file or provider key was loaded.")

---
### Step 2 — Offline LLM and Crew helpers
⬜ **NOTEBOOK ONLY**

`FakeLLM` implements the CrewAI 1.15 `BaseLLM.call()` interface, records every prompt and returns scripted replies. The first reply may request a tool and the next may provide a final structured answer. The network guard permits only the local loopback socket pair required by Python's Windows event loop.

In [ ]:
import json
import socket
from contextlib import contextmanager
from typing import Any
from unittest.mock import patch

import httpx
import requests
from crewai import Crew, Process, Task
from crewai.llms.base_llm import BaseLLM
from crewai.utilities.string_utils import sanitize_tool_name
from pydantic import PrivateAttr

class FakeLLM(BaseLLM):
    _replies: list[str] = PrivateAttr(default_factory=list)
    _calls: list[Any] = PrivateAttr(default_factory=list)

    def __init__(self, replies: list[str]):
        super().__init__(model="offline-fake-llm")
        self._replies, self._calls = list(replies), []

    def call(self, messages, tools=None, callbacks=None, available_functions=None,
             from_task=None, from_agent=None, response_model=None, **kwargs):
        self._calls.append(messages)
        if not self._replies:
            raise AssertionError("FakeLLM received an unexpected extra call.")
        return self._replies.pop(0)

    def prompt(self, index: int) -> str:
        return str(self._calls[index])

    @property
    def call_count(self) -> int:
        return len(self._calls)

    def supports_function_calling(self) -> bool:
        return False

    def supports_stop_words(self) -> bool:
        return True

    def get_context_window_size(self) -> int:
        return 8192

def tool_call(tool, **arguments) -> str:
    return ("Thought: I need approved local course information.\n"
            f"Action: {sanitize_tool_name(tool.name)}\n"
            f"Action Input: {json.dumps(arguments)}")

def final_answer(payload: dict[str, Any]) -> str:
    return ("Thought: I now know the final answer\n"
            f"Final Answer: {json.dumps(payload)}")

async def run_alone(agent, task):
    crew = Crew(agents=[agent], tasks=[task], process=Process.sequential,
                verbose=False, memory=False, share_crew=False, tracing=False)
    return crew, await crew.kickoff_async()

@contextmanager
def offline_network_guard():
    original_connect = socket.socket.connect

    def blocked(*args, **kwargs):
        raise AssertionError("External network access is forbidden in this workbook.")

    def local_event_loop_only(sock, address):
        host = address[0] if isinstance(address, tuple) and address else None
        if host in {"127.0.0.1", "::1"}:
            return original_connect(sock, address)
        return blocked(sock, address)

    with patch.object(socket, "create_connection", blocked), \
         patch.object(socket.socket, "connect", local_event_loop_only), \
         patch.object(requests.sessions.Session, "request", blocked), \
         patch.object(httpx.Client, "request", blocked), \
         patch.object(httpx.AsyncClient, "request", blocked):
        yield

---
### Step 3 — Real read-only Enrollment tool
🟩 **PRODUCTION EQUIVALENT**: `backend/tools/enrollment/course_information_tool.py`

The notebook imports the production `CourseInformationTool`; it does not create a substitute tool. This tool reads the approved local catalogue and has no repository or database-write dependency.

In [ ]:
from backend.agents.enrollment_agent import build_enrollment_agent
from backend.agents.schemas import EnrollmentExtraction
from backend.services.course_catalogue import CourseCatalogue
from backend.tools.enrollment.course_information_tool import CourseInformationTool

catalogue = CourseCatalogue()
course_tool = CourseInformationTool(catalogue)
assert course_tool.name == "approved_course_information"

### ✅ CHECKPOINT 1 — Direct tool tests
Call the production tool before building the agent. Test a valid topic, an unknown topic and an unsupported request. No Flask application or database is opened.

In [ ]:
approved = course_tool.run(topic="enrollment")
assert approved.source == "backend/data/course_information.json"
assert approved.approved_information["course"]["name"] == catalogue.course["name"]
assert approved.matched_faqs
assert not hasattr(course_tool, "_repository")

In [ ]:
unknown = course_tool.run(topic="topic-that-does-not-exist")
unsupported = course_tool.run(topic="private participant records")
for response in (unknown, unsupported):
    assert response.source == "backend/data/course_information.json"
    assert response.matched_faqs == []
    assert response.approved_information["course"]
print("CHECKPOINT 1 PASS: production tool is local, read-only and safely scoped.")

---
### Step 4 — Production Enrollment Agent
🟩 **PRODUCTION EQUIVALENT**: `backend/agents/enrollment_agent/__init__.py`

Import the working builder and give it only the read-only course tool. CrewAI 1.15.21 normalises an Agent's disabled memory value to `None`; the standalone Crew below explicitly uses `memory=False`.

In [ ]:
from crewai import Agent

wiring_agent = build_enrollment_agent(FakeLLM([]), [course_tool])
assert isinstance(wiring_agent, Agent)
assert wiring_agent.role == "Enrollment Agent"
assert "Extract only explicitly supplied" in wiring_agent.goal
assert wiring_agent.allow_delegation is False
assert wiring_agent.memory in {False, None}
assert [tool.name for tool in wiring_agent.tools] == ["approved_course_information"]
assert not hasattr(course_tool, "_repository")

---
### Step 5 — Run the Enrollment Agent alone
🟩 **PRODUCTION EQUIVALENT**: `CrewAIService.extract_enrollment()` creates the structured Task and `CrewAIService._run()` constructs the one-agent Crew.

The fictional message supplies only a course choice. It contains no identity or contact information, so every unrelated field must remain `None`.

In [ ]:
course_name = catalogue.course["name"]
runtime_tool = CourseInformationTool(catalogue)
structured_payload = {
    "extracted_fields": {"course": course_name},
    "missing_fields": [],
    "next_question": None,
    "ready_for_confirmation": False,
    "confirmation": "none",
    "correction_requested": False,
}
fake_llm = FakeLLM([
    tool_call(runtime_tool, topic="enrollment"),
    final_answer(structured_payload),
])
standalone_agent = build_enrollment_agent(fake_llm, [runtime_tool])
standalone_task = Task(
    description=("A fictional participant says: 'I would like to enrol in the "
                 f"{course_name}.' Use the approved course-information tool, "
                 "then extract only fields explicitly supplied in that sentence."),
    expected_output=("Valid EnrollmentExtraction JSON with no invented identity, "
                     "contact or intake values."),
    agent=standalone_agent, tools=[runtime_tool],
    output_pydantic=EnrollmentExtraction,
)

### ✅ CHECKPOINT 2 — Genuine CrewAI execution
The next cell makes a genuine `crew.kickoff_async()` call through `run_alone()`. It uses one real Agent, one real Task, `Process.sequential`, `memory=False` and the production Pydantic output contract.

In [ ]:
with offline_network_guard(), patch.object(runtime_tool, "_run", wraps=runtime_tool._run) as tool_run:
    standalone_crew, crew_result = await run_alone(standalone_agent, standalone_task)

assert len(standalone_crew.agents) == 1
assert len(standalone_crew.tasks) == 1
assert standalone_crew.process == Process.sequential
assert standalone_crew.memory is False
assert standalone_crew.tasks[0].agent is standalone_crew.agents[0]
assert [tool.name for tool in standalone_crew.tasks[0].tools] == [runtime_tool.name]

---
### Step 6 — Offline FakeLLM tool-call proof
⬜ **NOTEBOOK ONLY**, with the same test transferred to `backend/tests/enrollment_tests/test_enrollment_agent_isolated.py`.

The first FakeLLM reply selected the production tool. CrewAI executed it, inserted its observation into the second prompt, and accepted the second scripted reply as structured output.

In [ ]:
assert tool_run.call_count == 1
assert fake_llm.call_count == 2
assert "backend/data/course_information.json" in fake_llm.prompt(1)
assert "approved_information" in fake_llm.prompt(1)
assert isinstance(crew_result.pydantic, EnrollmentExtraction)
supplied = crew_result.pydantic.extracted_fields.model_dump(exclude_none=True)
assert supplied == {"course": course_name}
assert crew_result.pydantic.ready_for_confirmation is False

### ✅ CHECKPOINT 3 — Offline test evidence

The assertions cover the direct tool result, unknown and unsupported topics, Agent construction, tool attachment, delegation disabled, genuine kickoff, CrewAI-selected tool execution, tool output in the second prompt, Pydantic parsing, no invented participant fields and blocked external networking.

In [ ]:
checkpoint_3 = {
    "direct_tool": approved.source == "backend/data/course_information.json",
    "agent_wiring": standalone_agent.allow_delegation is False,
    "real_kickoff": isinstance(crew_result.pydantic, EnrollmentExtraction),
    "tool_called_once": tool_run.call_count == 1,
    "observation_returned": "approved_information" in fake_llm.prompt(1),
    "no_invented_fields": supplied == {"course": course_name},
    "no_provider_calls": fake_llm.call_count == 2,
}
assert all(checkpoint_3.values())
print("CHECKPOINT 3 PASS:", ", ".join(checkpoint_3))

---
### Step 7 — Deterministic application boundary
🟩 **PRODUCTION EQUIVALENT**: `backend/services/enrollment/validation.py` and `backend/services/enrollment/enrollment_service.py`.

CrewAI performs natural-language interpretation and returns `EnrollmentExtraction`. Deterministic Python validates DOB and age, NRIC/FIN checksum, email, mobile, course and intake values. Application services—not the LLM—control persistence, capacity allocation, confirmation, invoice creation and email queuing. The following checks use synthetic values and do not persist anything.

In [ ]:
from datetime import date
from backend.services.enrollment.validation import calculate_age, validate_field

assert calculate_age(date(2000, 5, 15), date(2026, 9, 29)) == 26
email, email_error = validate_field("email", "avery@example.com", catalogue)
mobile, mobile_error = validate_field("mobile_number", "91234567", catalogue)
assert email_error is mobile_error is None
assert mobile == "91234567"
print("Deterministic validation completed in memory; nothing was persisted.")

---
### Step 8 — Enrollment capability card
⬜ **NOTEBOOK ONLY**

This SkillSpec-style card documents what the Enrollment Agent can receive and produce. It is not added to the runtime because the current Flask router does not consume a SkillSpec registry.

In [ ]:
import re
from pydantic import BaseModel, Field

class EnrollmentSkillSpec(BaseModel):
    name: str
    label: str
    purpose: str
    keywords: str
    common_misspellings: list[str]
    required_data_groups: list[list[str]]
    optional_data: list[str] = Field(default_factory=list)
    allowed_context: list[str]
    read_only_tools: list[str]
    output_contract: str
    prohibited_actions: list[str]

enrollment_skill = EnrollmentSkillSpec(
    name="enrollment",
    label="course enrollment and continuation",
    purpose="Extract explicitly supplied enrollment details and corrections.",
    keywords=r"\b(?:enrol(?:l|ment)?|enrool|register|regster|sign up|apply|continue my application)\b",
    common_misspellings=["enrool", "regster", "enrollemnt"],
    required_data_groups=[["course"], ["preferred_intake_date"],
                          ["full_name", "nric", "date_of_birth", "email", "mobile_number"]],
    optional_data=["confirmation", "correction_requested"],
    allowed_context=["authorised conversation", "masked draft", "approved course data",
                     "deterministic validation results", "intake availability"],
    read_only_tools=["authorised_conversation_memory", "approved_course_information",
                     "current_enrollment_draft", "deterministic_enrollment_validation",
                     "course_intake_availability"],
    output_contract="EnrollmentExtraction",
    prohibited_actions=["database writes", "capacity allocation", "final confirmation",
                        "invoice creation", "email delivery", "inventing missing values"],
)
assert len(enrollment_skill.read_only_tools) == 5

In [ ]:
positive = ["I want to enrol in the dental course", "Please register me", "Can I sign up?"]
misspelled = ["I want to enrool", "Please regster me"]
negative = ["What is the course fee?", "Can I use UTAP?", "Hello there"]
for message in positive + misspelled:
    assert re.search(enrollment_skill.keywords, message.casefold()), message
for message in negative:
    assert not re.search(enrollment_skill.keywords, message.casefold()), message
print("Capability keyword checks passed.")

---
### Step 9 — Framework integration mapping

| Workbook element | Production project |
|---|---|
| Approved course tool | `backend/tools/enrollment/course_information_tool.py` |
| Five-tool factory | `backend/tools/enrollment/__init__.py::build_enrollment_tools()` |
| Enrollment Agent | `backend/agents/enrollment_agent/__init__.py::build_enrollment_agent()` |
| Structured schema | `backend/agents/schemas.py::EnrollmentExtraction` |
| Structured Task | `backend/services/crewai_service.py::CrewAIService.extract_enrollment()` |
| Crew execution | `backend/services/crewai_service.py::CrewAIService._run()` |
| Routing | `backend/services/chat_service.py::ChatService.respond()` |
| Validation/state machine | `backend/services/enrollment/validation.py`, `backend/services/enrollment/enrollment_service.py` |
| Persistent memory | `backend/services/conversation_repository.py`, `backend/models/` |
| Offline isolated test | `backend/tests/enrollment_tests/test_enrollment_agent_isolated.py` |

This adapted evidence workbook imports the existing production implementation to prevent notebook/source drift. It recreates and proves the independent Tool → Agent → Task → Crew → kickoff method; it does not claim that the production Agent was originally authored here. The final application remains independent of this file.

---
### Step 10 — Transfer and pull-request checklist

- [x] Direct production tool tests pass.
- [x] Agent role, tool and delegation wiring pass.
- [x] FakeLLM causes CrewAI to select and call the tool exactly once.
- [x] The tool observation reaches the second FakeLLM prompt.
- [x] The result parses as `EnrollmentExtraction`.
- [x] No unrelated participant fields are invented.
- [x] CrewAI contracts, Enrollment tools and the complete Enrollment workflow test module pass locally.
- [x] No `.env`, credential value, real identity or participant record is included in this evidence.
- [x] Notebook outputs and execution counts are cleared before review.
- [x] No project database, invoice, PDF, workbook or email row was changed by this evidence run.
- [x] API schema, persistence, validation, invoice and email behaviour remain unchanged; the existing FAQ-to-enrollment continuation behaviour is restored for the deterministic course-fact route.
- [ ] The remote GitLab pipeline and merge request must be confirmed after push.

The transferable automated evidence lives in `backend/tests/enrollment_tests/test_enrollment_agent_isolated.py`; notebook-only teaching helpers do not become production dependencies. The complete `backend/tests/enrollment_tests/test_enrollment.py` module is included in the offline GitLab job rather than being filtered to a smaller passing subset.